Make sure the right schema is used

In [0]:
USE CATALOG sac;
USE SCHEMA customer_service;

In [0]:
select zone, count(*) from customer group by zone

zone,count(*)
Schleswig-Holstein,111
Sachsen,151
Brandenburg,101
Nordrhein-Westfalen,521
Berlin,46
Hamburg,33
Thüringen,37


# Gold Tables
average customer

In [0]:
CREATE OR REPLACE VIEW average_customer AS
SELECT
    zone,
    COUNT(*) AS amount_customers,
    ROUND(AVG(monthly_bill), 2) AS avg_monthly_bill,
    ROUND(AVG(speed_tier_mbps), 0) AS avg_speed_tier,
    ROUND(AVG(data_usage_gb_last_month), 2) AS avg_data_usage
FROM
    customer
GROUP BY
    zone;

tickets per customer

In [0]:
CREATE OR REPLACE VIEW customer_connection_ticket_count AS
SELECT
    c.customer_id,
    COUNT(DISTINCT l.timestamp) AS connection_fails,
    COUNT(DISTINCT s.ticket_id) AS tickets,
    COUNT(DISTINCT ca.session_id) AS chats,
    ch.churned as churned
FROM
    customer c
        JOIN ticket s
            ON c.customer_id = s.customer_id
        LEFT JOIN log l
            ON c.customer_id = l.customer_id
            AND l.issue_detected != 'none'
        LEFT JOIN churn ch
            ON c.customer_id = ch.customer_id
        LEFT JOIN chat ca
            ON c.customer_id = ca.customer_id
GROUP BY
    c.customer_id, ch.churned;

churned customer

In [0]:
CREATE OR REPLACE VIEW churned_customer_details AS
SELECT
    c.customer_id,
    c.speed_tier_mbps,
    c.monthly_bill,
    COUNT(DISTINCT l.timestamp) AS connection_fails,
    COUNT(DISTINCT s.ticket_id) AS tickets,
    COUNT(DISTINCT ca.session_id) AS chats
FROM
    customer c
        LEFT JOIN ticket s
            ON c.customer_id = s.customer_id
        LEFT JOIN log l
            ON c.customer_id = l.customer_id
            AND l.issue_detected != 'none'
        LEFT JOIN chat ca
            ON c.customer_id = ca.customer_id
        JOIN churn ch
            ON c.customer_id = ch.customer_id
WHERE
    ch.churned = true
GROUP BY
    c.customer_id,
    c.speed_tier_mbps,
    c.monthly_bill;

location detail

In [0]:
CREATE OR REPLACE VIEW location_detail AS
WITH revenue_per_location AS (
    SELECT
        zone,
        SUM(monthly_bill) AS revenue
    FROM
        customer
    GROUP BY
        zone
),
issues_per_location AS (
    SELECT
        c.zone,
        COUNT(
            CASE
                WHEN l.issue_detected != 'none' THEN 1
            END
        ) AS issue_count
    FROM
        customer c
            LEFT JOIN log l
                ON c.customer_id = l.customer_id
    GROUP BY
        c.zone
)
SELECT
    c.zone,
    COUNT(DISTINCT c.customer_id) AS customer_count,
    ROUND(r.revenue / 1000, 2) AS revenue_in_t,
    i.issue_count AS issue_count,
    COUNT(DISTINCT t.ticket_id) AS ticket_count,
    COUNT(DISTINCT ch.session_id) AS chat_count
FROM
    customer c
        LEFT JOIN revenue_per_location r
            ON c.zone = r.zone
        LEFT JOIN issues_per_location i
            ON c.zone = i.zone
        LEFT JOIN ticket t
            ON c.customer_id = t.customer_id
        LEFT JOIN chat ch
            ON c.customer_id = ch.customer_id
GROUP BY
    c.zone,
    r.revenue,
    i.issue_count;

average connection quality

In [0]:
CREATE OR REPLACE VIEW average_connection_quality AS
SELECT
    c.zone,
    ROUND(AVG(l.speed_measured_mbps), 0) AS avg_speed,
    ROUND(AVG(l.packet_loss_percent), 2) AS avg_packet_loss,
    ROUND(AVG(l.latency_ms), 2) AS avg_latency,
    ROUND(AVG(l.downtime_minutes), 2) AS avg_downtime,
    ROUND(AVG(l.connection_drops_count), 2) AS avg_connection_drops,
    COUNT(
        CASE
            WHEN l.issue_detected != 'none' THEN 1
            ELSE 0
        END
    ) AS count_issues
FROM
    log l
        LEFT JOIN customer c
            ON l.customer_id = c.customer_id
GROUP BY
    zone;

chat issues

In [0]:
CREATE OR REPLACE VIEW chat_issues AS
SELECT
    c.classification,
    m.sentiment,
    COUNT(m.sentiment) AS count,
    FIRST(c.comment) AS exmp_comment
FROM
    chat c
    LEFT JOIN message m
WHERE
    classification IS NOT NULL
    AND m.speaker = 'customer'
GROUP BY
    c.classification,
    m.sentiment
ORDER BY
    count DESC

sentiment for agent

In [0]:
CREATE OR REPLACE VIEW sentiment_for_agent AS
SELECT
    CONCAT(a.first_name, ' ', a.last_name) AS agent_name,
    m.sentiment,
    COUNT(m.sentiment) AS count
FROM
    chat c
        JOIN message m
            ON c.session_id = m.session_id
            AND m.speaker = 'customer'
        LEFT JOIN agent a
            ON c.agent_id = a.agent_id
GROUP BY
    agent_name,
    m.sentiment;

# Show tables

In [0]:
SELECT * FROM average_customer;

zone,amount_customers,avg_account_age,avg_monthly_bill,avg_speed_tier,avg_data_usage
Mecklenburg-Vorpommern,78,36.6,70.49,238.0,241.78
Rheinland-Pfalz,30,29.68,69.43,242.0,436.1
Niedersachsen,75,36.55,73.39,330.0,283.21
Schleswig-Holstein,20,33.77,70.12,228.0,246.16
Sachsen,67,30.76,77.76,368.0,274.28
Bayern,217,33.87,71.27,289.0,311.33
Brandenburg,84,30.17,72.54,332.0,285.3
Hamburg,5,41.36,70.99,170.0,111.82
Nordrhein-Westfahlen,94,32.47,71.48,282.0,300.65
Baden-Württemberg,104,36.13,70.16,285.0,297.49


In [0]:
SELECT * FROM customer_connection_ticket_count ORDER BY tickets DESC LIMIT 20;

customer_id,connection_fails,tickets,chats,churned
CUST_00971,2,3,0,false
CUST_00865,5,3,2,false
CUST_00739,6,3,2,false
CUST_00167,3,2,0,false
CUST_00938,2,2,0,true
CUST_00567,1,2,0,false
CUST_00666,4,2,0,false
CUST_00810,5,2,0,false
CUST_00464,5,2,0,false
CUST_00937,3,2,2,false


In [0]:
SELECT * FROM churned_customer_details ORDER BY tickets DESC LIMIT 20;

customer_id,account_age_months,speed_tier_mbps,monthly_bill,connection_fails,tickets,chats
CUST_00127,58.2,50,49.43,8,2,2
CUST_00021,65.3,200,75.91,5,2,0
CUST_00628,24.8,50,48.66,6,2,1
CUST_00608,26.1,200,79.98,7,2,0
CUST_00604,26.4,200,79.7,3,2,2
CUST_00938,4.1,200,74.06,2,2,0
CUST_00159,56.1,1000,111.02,5,1,0
CUST_00376,41.6,200,73.61,16,1,2
CUST_00446,36.9,1000,119.24,4,1,0
CUST_00344,43.7,200,77.97,2,1,0


In [0]:
SELECT * FROM location_detail ORDER BY customer_count DESC;

zone,customer_count,revenue_in_t,issue_count,ticket_count,chat_count
Bayern,217,15.47,4143,100,103
Baden-Württemberg,104,7.3,2070,35,46
Nordrhein-Westfahlen,94,6.72,1857,44,51
Brandenburg,84,6.09,1485,35,21
Thüringen,84,5.95,1521,45,46
Mecklenburg-Vorpommern,78,5.5,1353,28,42
Niedersachsen,75,5.5,1377,32,33
Sachsen-Anhalt,73,5.33,1380,28,41
Sachsen,67,5.21,1272,23,36
Hessen,47,3.58,906,10,30


In [0]:
SELECT * FROM average_connection_quality;

zone,avg_speed,avg_packet_loss,avg_latency,avg_downtime,avg_connection_drops,count_issues
Niedersachsen,287.0,2.26,38.17,2.03,0.94,7737
Thüringen,339.0,2.14,36.82,1.83,0.88,9045
Saarland,76.0,2.7,43.76,2.9,0.85,447
Bayern,250.0,2.19,37.78,2.06,1.04,22824
Schleswig-Holstein,244.0,2.14,39.23,3.25,1.27,2007
Bremen,84.0,1.81,35.69,2.8,1.04,276
Nordrhein-Westfahlen,261.0,2.3,38.81,1.54,0.92,9687
Hessen,242.0,2.3,38.17,1.24,0.8,5037
Sachsen,243.0,2.16,37.56,2.86,1.1,7029
Hamburg,390.0,1.77,39.66,2.36,0.75,531


In [0]:
SELECT * FROM chat_issues order by classification, sentiment;

classification,sentiment,count,exmp_comment
CONNECTION ISSUES,Negativ,26304,Kabel ist gebrochen
CONNECTION ISSUES,Neutral,53184,VPN-Client und Tunnelstabilität
CONNECTION ISSUES,Positiv,10560,VPN-Client und Tunnelstabilität
CONNECTION ISSUES,Unbekannt,11904,VPN-Client und Tunnelstabilität
OTHER,Negativ,822,Datumsproblem stört die Validierung der SSL-Zertifikate
OTHER,Neutral,1662,Datumsproblem stört die Validierung der SSL-Zertifikate
OTHER,Positiv,330,Datumsproblem stört die Validierung der SSL-Zertifikate
OTHER,Unbekannt,372,Datumsproblem stört die Validierung der SSL-Zertifikate
PRICE,Negativ,31236,Monatliche Rechnung 40 Euro
PRICE,Neutral,63156,Monatliche Rechnung beträgt 40 Euro


In [0]:
SELECT * FROM sentiment_for_agent ORDER BY agent_name, sentiment LIMIT 20;

agent_name,sentiment,count
Anna Schmidt,Negativ,35
Anna Schmidt,Neutral,31
Anna Schmidt,Positiv,6
Anna Schmidt,Unbekannt,6
Ben Neumann,Negativ,31
Ben Neumann,Neutral,27
Ben Neumann,Positiv,4
Ben Neumann,Unbekannt,8
David Wolf,Negativ,39
David Wolf,Neutral,20
